In [ ]:
# bootstrap: Colab clone + local import of `thinklab` (auto-inserted)
import sys, pathlib
if "google.colab" in sys.modules:
    import os, subprocess
    _slug = "aniryou/full-stack-agentic-engineer"
    _repo = pathlib.Path("/content/full-stack-agentic-engineer")
    if not _repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_repo)], check=True)
    os.chdir(_repo / "00-foundations/rl-and-thinking-models/thinking-lab")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
_r = pathlib.Path.cwd().resolve()
while _r != _r.parent and not (_r / "thinklab").exists():
    _r = _r.parent
if str(_r) not in sys.path:
    sys.path.insert(0, str(_r))
del _r

# 05 · RL rollouts with an inference engine: the bookkeeping of one GRPO step

**Tier:** T1 — vLLM generates G = 8 rollouts for each of 8 prompts with `Qwen/Qwen2.5-0.5B-Instruct`
(TRL's quick-start model), a verifier scores them, and transformers takes one GRPO step on a T4 (the
cell at the end runs only with a GPU, `vllm` and `transformers`; `deploy/any-gpu/rl_step.sh` wraps
it). T0 (default) — the same bookkeeping on the tiny transformer's rollouts: with torch they are
generated now by a bfloat16 "engine" copy and re-scored by a float32 "trainer" copy; without torch
a recorded set is used (illustrative).

## The one-minute version

* In RL for LLMs the **rollout is an inference workload**. G completions per prompt, generated by
  an engine inside the training loop, and the phase that dominates step time: verl's docs report
  about 70% of total time for DAPO training of a 32B model. Everything the serving layers taught
  applies: batching, KV capacity, long tails (PRIMER §8 "The RL training stack in brief").
* The trainer's bookkeeping per step: group rollouts by prompt → verify → group-normalised
  advantages → drop groups with no signal (DAPO's dynamic sampling) → recompute log-probs with
  the trainer's weights → correct for the **train–inference mismatch** (the engine's kernels and
  precision never match the trainer's exactly) → loss → update → **push the weights back** to the
  engine.
* Two systems costs follow. A synchronous rollout batch waits for its longest completion, so
  heavy-tailed thinking lengths leave GPUs idle, which is why one-step-off and fully async RL
  exist. And the weight sync is every parameter, every step, unless only deltas are sent.

In [ ]:
import json, math, statistics
from importlib import resources
from thinklab import engine, env
from thinklab.report import table
from thinklab.rollout import (Rollout, dynamic_sampling, frac_zero_std, group_advantages, group_by_prompt, is_ratios,
                              rollout_phase, soft_overlong, trl_grpo_config, weight_sync_bytes)

print(env.describe())
if env.has_torch():
    from thinklab.tinyrl import train as T
    model, task = T.warm_start(T.TinyRLConfig(), log=lambda *a: None)       # SFT warm-up only, ~30 s on a CPU
    RAW = T.make_rollouts(model, task, prompts=8, generations=8, seed=0)
    LABEL = "MEASURED now: bf16 engine copy vs fp32 trainer copy of the tiny transformer"
else:
    data = json.loads(resources.files("thinklab.data").joinpath("tinyrl_recorded_rollouts.json").read_text())
    RAW, LABEL = data["rollouts"], data["source"]
ROLLOUTS = [Rollout(r["prompt_id"], r["length"], r["reward"], r["sampler_logps"], r["trainer_logps"]) for r in RAW]
print(LABEL, "|", len(ROLLOUTS), "rollouts")
groups = group_by_prompt(ROLLOUTS)
print(table([{"prompt": k, "rewards": "".join(str(int(r.reward)) for r in g), "mean": round(statistics.fmean(r.reward for r in g), 3),
              "lengths": " ".join(str(r.tokens) for r in g)} for k, g in groups.items()], title="8 prompts x G = 8 rollouts"))

## Exercise 5.1 — from rollouts to a training batch

Group the rollouts by prompt and return `(advantages, kept)`:

* `advantages`: prompt id → the list of group-normalised advantages, `(r − mean) / (std + 1e-4)` with
  Bessel's std, in rollout order;
* `kept`: the prompt ids that DAPO's dynamic sampling keeps, the groups whose rewards are *not*
  all equal. The others contribute zero gradient, and a trainer would replace them with fresh
  prompts.

In [ ]:
def grpo_batch(rollouts: list) -> tuple:
    groups = {}
    for r in rollouts:
        groups.setdefault(r.prompt_id, []).append(r.reward)
    adv, kept = {}, []
    for pid, rs in groups.items():
        m = sum(rs) / len(rs)
        s = math.sqrt(sum((x - m) ** 2 for x in rs) / (len(rs) - 1)) if len(rs) > 1 else 0.0
        adv[pid] = [(x - m) / (s + 1e-4) for x in rs]
        if s > 0:
            kept.append(pid)
    return adv, kept

In [ ]:
adv, kept = grpo_batch(ROLLOUTS)
for pid, g in groups.items():
    assert all(abs(a - b) < 1e-9 for a, b in zip(adv[pid], group_advantages([r.reward for r in g])))
assert sorted(kept) == sorted(dynamic_sampling(groups))
toy = [Rollout("a", 3, 1.0), Rollout("a", 3, 0.0), Rollout("b", 3, 1.0), Rollout("b", 3, 1.0)]
assert grpo_batch(toy)[1] == ["a"] and grpo_batch(toy)[0]["b"] == [0.0, 0.0]
print(f"✅ {len(kept)} of {len(groups)} groups carry signal (frac_reward_zero_std = {frac_zero_std(groups):.2f})")

## Exercise 5.2 — the train–inference mismatch, and TRL's default correction

The engine reports the log-probability of each token it sampled (`sampler_logps`). The trainer
recomputes them with its own weights and kernels (`trainer_logps`). They should be equal and
are not. Implement TRL's default correction, `vllm_importance_sampling_mode="sequence_mask"`:
one ratio per sequence, `exp(Σ (trainer − sampler))`, kept if it lies in `[c_min, c_max]` and set
to 0 otherwise (the sequence is masked out of the loss). Return the per-token weights (the same
value for every token of the sequence). TRL's default `c_max` is 3.0 with no lower bound.

In [ ]:
def seq_mask_weights(sampler: list, trainer: list, c_max: float = 3.0, c_min: float | None = None) -> list:
    ratio = math.exp(sum(t - s for s, t in zip(sampler, trainer)))
    lo = -math.inf if c_min is None else c_min
    w = ratio if lo <= ratio <= c_max else 0.0
    return [w] * len(sampler)

In [ ]:
assert seq_mask_weights([-1.0, -1.0], [-1.0, -1.0]) == [1.0, 1.0]
assert seq_mask_weights([-2.0], [0.0]) == [0.0]                          # e^2 = 7.4 > 3: masked
assert seq_mask_weights([-1.0], [-1.5], c_min=0.7) == [0.0]              # e^-0.5 = 0.61 < 0.7: masked
for r in ROLLOUTS:
    assert all(abs(a - b) < 1e-9 for a, b in zip(seq_mask_weights(r.sampler_logps, r.trainer_logps),
                                                   is_ratios(r.sampler_logps, r.trainer_logps, "sequence_mask")))
diffs = [abs(t - s) for r in ROLLOUTS for s, t in zip(r.sampler_logps, r.trainer_logps)]
w = [seq_mask_weights(r.sampler_logps, r.trainer_logps)[0] for r in ROLLOUTS]
print(f"✅ [{LABEL.split(':')[0]}] mean |log p_trainer - log p_sampler| = {statistics.fmean(diffs):.4f} per token "
      f"(max {max(diffs):.4f}); sequence weights in [{min(w):.3f}, {max(w):.3f}]")

On the tiny model the only difference between the two copies is bfloat16 versus float32, and the
mismatch is already visible. With a real engine it adds different attention kernels, batch-size-
dependent reductions, sampling tricks such as top-k or FP8 KV, and weights one step stale. TRL logs
it as `sampling/sampling_logp_difference/mean`.

## Worked example: the rollout phase is a serving problem with a straggler

A synchronous GRPO step generates the whole batch, then trains. The batch shrinks as completions
finish, and the phase ends only when the *longest* one does. With thinking-length tails that
leaves most sequence slots idle at the end. Below, 512 rollouts (64 prompts × G = 8) with lengths
from the simulated thinking model, decoded on the engine emulator's Qwen3-0.6B/T4 step model at
the batch still running (**simulated**).

In [ ]:
from thinklab.workload import sample_lengths
from thinklab.thinking.evalset import make_evalset
PROF = engine.profile("t4-qwen3-0.6b")
qs = [p.prompt for p in make_evalset(64, seed=9) for _ in range(8)]
SAMPLES = sample_lengths(qs, seed=0)                              # (reasoning, answer, correct) per rollout
lens = [min(r + a, 8000) for r, a, _ in SAMPLES]
avg_ctx = 60 + statistics.fmean(lens) / 2
step = lambda b: PROF.decode_step_s(b, b * avg_ctx)              # noqa: E731 — the roofline step at batch b
ph = rollout_phase(lens, step, overlap_training_s=20.0)
print(table([{"rollouts": len(lens), "mean tokens": round(ph["mean"]), "longest": ph["longest"],
              "phase s": round(ph["phase_s"], 1), "idle slot share": round(ph["idle_share"], 2),
              "sync step s (+20 s train)": round(ph["sync_step_s"], 1),
              "one-step-off step s": round(ph["overlapped_step_s"], 1)}], title="[SIMULATED] one rollout phase"))

## Exercise 5.3 — pick a generation cap with DAPO's overlong shaping

A lower `max_completion_length` trims the straggler tail above, but DAPO's soft overlong
penalty (rl-core notebook 03, exercise 3.5; `soft_overlong` here) charges completions in the last
`cache` tokens before the cap, and −1 beyond it, including ones that would have reached a correct
answer given room. The cap is a reward-shaping decision, not only a systems one. For each cap in
`CAPS`, with DAPO's proportions (cache = cap / 5, as 4,096 of 20,480), `overlong_view` returns the
share of these 512 simulated rollouts longer than the cap (`truncated`) and the share of the
rollouts that would have been correct (the simulated model's uncapped outcome) that get a penalty
below 0 (`hit_correct`). Then set `cap` to the smallest cap in `CAPS` that penalises at most 5% of
the would-be-correct rollouts.

In [ ]:
CAPS = (1024, 2048, 4096, 6144, 8192)
RAW_LENS, CORRECT = [r + a for r, a, _ in SAMPLES], [c for _, _, c in SAMPLES]

def overlong_view(lengths: list, correct: list, cap: int) -> dict:
    pen = [soft_overlong(n, cap, cap // 5) for n in lengths]
    return {"cap": cap, "truncated": statistics.fmean(n > cap for n in lengths),
            "hit_correct": statistics.fmean(x < 0 for x, c in zip(pen, correct) if c)}

cap = None
cap = min(v["cap"] for v in (overlong_view(RAW_LENS, CORRECT, c) for c in CAPS) if v["hit_correct"] <= 0.05)

In [ ]:
v = overlong_view([100, 90, 50, 120], [True, True, True, False], 100)       # cache 20: 100 → −1, 90 → −0.5, 50 → 0
assert v == {"cap": 100, "truncated": 0.25, "hit_correct": 2 / 3}
views = [overlong_view(RAW_LENS, CORRECT, c) for c in CAPS]
print(table([{k: (round(x, 3) if isinstance(x, float) else x) for k, x in v.items()} for v in views],
            title="[SIMULATED] DAPO's soft overlong penalty at cache = cap / 5"))
assert cap == min(v["cap"] for v in views if v["hit_correct"] <= 0.05) == 6144
tight = next(v for v in views if v["cap"] == 2048)
print(f"✅ cap {cap:,}: at 2,048 the penalty would push against {tight['hit_correct']:.0%} of the reasoning that was "
      "on its way to a right answer, and teach the policy to stop early where it should not")

## Exercise 5.4 — the idle share of a synchronous rollout batch

With a constant step time (so only the *shape* of the length distribution matters), all
sequences start together and each occupies its slot until it finishes. The batch reserves
`n × max(lengths)` slot-steps and uses `Σ lengths` of them. Return the idle share,
`1 − Σ lengths / (n × max)`. Then answer: for the lengths above, what share of the batch's slots
sits idle?

In [ ]:
def idle_share(lengths: list) -> float:
    return 1 - sum(lengths) / (len(lengths) * max(lengths))

In [ ]:
assert idle_share([10, 10, 10]) == 0.0 and abs(idle_share([100, 200, 1000]) - (1 - 1300 / 3000)) < 1e-12
assert abs(idle_share(lens) - rollout_phase(lens, lambda b: 1.0)["idle_share"]) < 1e-9
print(f"✅ {idle_share(lens):.0%} of the slot-steps are idle while the tail finishes; "
      f"a thinking budget (or async rollouts) attacks exactly this")

## Exercise 5.5 — what goes back to the engine every step

After each optimizer step the engine needs the new weights. Return the bytes of a full sync, the
bytes of a delta sync when only `changed` of the bf16 weight bytes change (verl's measurement:
over 99% unchanged step over step), and the seconds each takes at `link_gbs` gigabytes per second.
Use Qwen2.5-0.5B-Instruct's 494 M parameters.

In [ ]:
def sync_cost(params: float, changed: float, link_gbs: float, bytes_per_param: float = 2.0) -> dict:
    full = params * bytes_per_param
    delta = full * changed
    return {"full_bytes": full, "delta_bytes": delta, "full_s": full / (link_gbs * 1e9), "delta_s": delta / (link_gbs * 1e9)}

In [ ]:
c = sync_cost(494e6, 0.02, 16.0)                              # ~PCIe Gen4 x16 order of magnitude (verify for your box)
assert abs(c["full_bytes"] - 988e6) < 1 and abs(c["delta_bytes"] - 19.76e6) < 1
assert abs(c["full_s"] - weight_sync_bytes(494e6) / 16e9) < 1e-12 and c["delta_s"] < c["full_s"] / 49
big = sync_cost(32e9, 0.02, 16.0)
print(f"✅ 0.5B: {c['full_bytes'] / 1e9:.2f} GB per step ({c['full_s']:.3f} s) vs {c['delta_bytes'] / 1e6:.0f} MB delta; "
      f"a 32B policy: {big['full_bytes'] / 1e9:.0f} GB = {big['full_s']:.1f} s per step at 16 GB/s")

## On a real GPU (T1): vLLM as the rollout generator for one GRPO step

```bash
pip install -q "vllm==0.30.0" "transformers>=4.56.2"      # optional: "trl==1.14.0" for GRPOTrainer
python -m thinklab rl-step --model Qwen/Qwen2.5-0.5B-Instruct --prompts 8 -g 8
```

`thinklab.rollout.one_grpo_step` loads the model in vLLM (`gpu_memory_utilization=0.3`, so the
trainer copy fits beside it on a 15 GB T4 (verify)), generates 8 × 8 rollouts with `logprobs=0`
(the sampled token's log-prob at every position), scores them with the eval-set verifier, computes
advantages, recomputes log-probs with a float32 transformers copy, applies a sequence-level
importance weight masked above 3 (`is_ratios(..., mode="sequence_mask")`, the function from
Exercise 5.2 and TRL's default), and takes one SGD step. It reports rollout seconds against training
seconds, the reward, the zero-std share and the sampler/trainer log-prob gap. It does not push the
weights back into vLLM. TRL's colocate mode does that every step; the configuration below is
where to start with TRL on a T4.

In [ ]:
print(json.dumps(trl_grpo_config("T4"), indent=1))
if env.gpu_name() and env.has_vllm():
    from thinklab.rollout import one_grpo_step
    one_grpo_step("Qwen/Qwen2.5-0.5B-Instruct", n_prompts=8, n=8)
else:
    print("T0: no GPU with vLLM here; the rollout bookkeeping above is the part that transfers. "
          "On a T4: python -m thinklab rl-step (deploy/any-gpu/rl_step.sh)")

## In a design review

**Two minutes:** "Our RL loop is an inference service with a trainer attached. Each step,
vLLM generates G = 8 completions per prompt and returns their log-probs, a verifier scores them,
the trainer normalises rewards within each group, and groups where every sample agrees are
dropped because they carry no gradient. The trainer recomputes log-probs, and the gap to the
engine's (kernels, precision, stale weights) is corrected by a sequence-level importance weight,
masked above 3. Rollouts are most of the step time. A synchronous batch waits for its longest
completion, so with thinking-length tails most slots idle at the end; budgets, overlong
penalties and one-step-off rollouts attack that. The new weights go back to the engine every
step: a full copy is the model size. Delta sync sends ~50× fewer bytes when 2% of the weights
change (Exercise 5.5), but verl measured 1.3–21× less wall time, because diffing, encoding and
fixed overheads remain (PRIMER §8 "The RL training stack in brief", verify). The serving skills
transfer directly: batch shape, KV capacity, tail latency."

**Drill 1.** *Half our groups have all-correct rewards. Problem?* Those prompts are too easy for
the current policy and contribute no gradient. Filter them (dynamic sampling), raise difficulty, or
accept a smaller effective batch. TRL logs the share as `frac_reward_zero_std`.

**Drill 2.** *Why not trust vLLM's log-probs as the old policy?* They come from different kernels,
batch shapes and sometimes stale weights, so they differ from the trainer's. The trainer recomputes
them, and the ratio corrects the update (TRL's `vllm_importance_sampling_correction`, on by
default).

**Drill 3.** *Rollouts are 70% of step time and adding GPUs doesn't help. Why?* The phase is bound
by the longest completions, not by throughput. Overlap generation with training (one-step-off or
fully async RL, which trade some staleness), cap lengths (budgets, overlong shaping), or pack new
prompts into the freed slots.